In [ ]:
# ================================================================================================
# SPX NOON DIRECTION ENGINE — LIVE / DAILY PRODUCTION VERSION
#
# FREE DATA SOURCE:
#   Yahoo Finance ^GSPC 1-minute OHLC
#
# PURPOSE:
#   Download/cache SPX M1 data and calculate TODAY'S frozen noon signal.
#
# SIGNAL:
#   BULLISH  -> Sell PUT credit spread
#   BEARISH  -> Sell CALL credit spread
#   NO_TRADE -> Skip
#
# FROZEN ENGINE:
#   Snapshot:       11:58 ET
#   Morning:        09:30-11:58 ET
#   10m EMA/REG:    anchor 09:30, partial current bar
#   ROC5 15m:       anchor 09:29, partial, shift 5
#   ROC3 30m:       anchor 09:29, partial, shift 3
#   ATR14:          daily OHLC 09:30-15:58 ET
#                   simple mean previous 14 completed TRs
#   M1 EMA20:       continuous regular-session EMA20
#
# VALIDATION:
#   Frozen historical dates:       245
#   Period:                        2025-06-02 through 2026-06-16
#   Trades:                        206
#   Wins / Losses:                 188 / 18
#   Win rate:                      91.26%
#   Profit factor:                 2.41
#   Historical net:                $3,250
#   Max drawdown:                  $400
#   Production reconstruction:     245/245 = 100%
#   15-21 session warm-up test:    245/245 = 100%
#
# IMPORTANT:
#   Yahoo only retains 1-minute history for a limited period.
#   This script therefore caches every download in Google Drive.
#
# ================================================================================================


!pip -q install --upgrade yfinance


import os
import warnings

from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
import yfinance as yf

from IPython.display import display, HTML


warnings.filterwarnings(
    "ignore"
)


# ================================================================================================
# GOOGLE DRIVE
# ================================================================================================

try:

    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False
    )

    CACHE_DIR = Path(
        "/content/drive/MyDrive/TradingBacktests/SPX_Production"
    )

except Exception:

    CACHE_DIR = Path(
        "./SPX_Production"
    )


CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CACHE_FILE = (
    CACHE_DIR /
    "SPX_YAHOO_M1_CACHE.csv"
)


print()
print("=" * 110)
print("SPX PRODUCTION ENGINE")
print("=" * 110)

print(
    f"Cache:\n{CACHE_FILE}"
)


# ================================================================================================
# CONFIGURATION
# ================================================================================================

SYMBOL = "^GSPC"

NY_TZ = "America/New_York"

SESSION_START = 570
SESSION_END = 959

SIGNAL_LAST_MINUTE = 718

ATR_SESSION_START = 570
ATR_SESSION_END = 958

POSITION_LIMIT = 0.90

DIST_HIGH_THRESHOLD = 0.54365124
EMA20_DIST_THRESHOLD = -0.04307731


# ================================================================================================
# FROZEN VALIDATION RECORD
#
# These are descriptive historical results from the frozen validation.
# They are NOT recalculated or optimized by the production script.
# ================================================================================================

VALIDATION = {

    "period":
        "2025-06-02 → 2026-06-16",

    "dates":
        245,

    "trades":
        206,

    "wins":
        188,

    "losses":
        18,

    "win_rate":
        91.2621,

    "profit_factor":
        2.40845,

    "net_profit":
        3250.00,

    "max_drawdown":
        400.00,

    "bullish":
        109,

    "bearish":
        97,

    "no_trade":
        39,

    "reconstruction":
        "245 / 245 (100%)",

    "warmup_21":
        "245 / 245 (100%)",

    "warmup_18":
        "245 / 245 (100%)",

    "warmup_16":
        "245 / 245 (100%)",

    "warmup_15":
        "245 / 245 (100%)"
}


# ================================================================================================
# HELPERS
# ================================================================================================

def sign_value(x):

    if pd.isna(x):
        return 0

    if x > 0:
        return 1

    if x < 0:
        return -1

    return 0


def regression_slope(values):

    values = np.asarray(
        values,
        dtype=float
    )

    if len(values) < 2:
        return np.nan

    if np.isnan(values).any():
        return np.nan

    x = np.arange(
        len(values),
        dtype=float
    )

    return float(
        np.polyfit(
            x,
            values,
            1
        )[0]
    )


# ================================================================================================
# DOWNLOAD YAHOO M1
# ================================================================================================

def download_yahoo_m1():

    print()
    print("=" * 110)
    print("DOWNLOADING RECENT SPX M1 DATA")
    print("=" * 110)

    now_ny = pd.Timestamp.now(
        tz=NY_TZ
    )

    end_date = (
        now_ny.date()
        +
        timedelta(days=1)
    )

    start_date = (
        now_ny.date()
        -
        timedelta(days=29)
    )

    chunks = []

    cursor = start_date

    while cursor < end_date:

        chunk_end = min(
            cursor + timedelta(days=7),
            end_date
        )

        print(
            f"Downloading {cursor} -> {chunk_end}"
        )

        try:

            df = yf.download(
                SYMBOL,
                start=str(cursor),
                end=str(chunk_end),
                interval="1m",
                auto_adjust=False,
                progress=False,
                prepost=False,
                threads=False
            )

        except Exception as e:

            print(
                f"  Download error: {e}"
            )

            cursor = chunk_end
            continue


        if df is None or len(df) == 0:

            print(
                "  No bars returned."
            )

            cursor = chunk_end
            continue


        if isinstance(
            df.columns,
            pd.MultiIndex
        ):

            df.columns = [
                c[0]
                for c in df.columns
            ]


        df = df.reset_index()


        time_candidates = [
            "Datetime",
            "Date",
            "datetime",
            "date"
        ]


        time_col = None


        for candidate in time_candidates:

            if candidate in df.columns:

                time_col = candidate
                break


        if time_col is None:

            raise RuntimeError(
                "Could not identify Yahoo timestamp column."
            )


        df = df.rename(
            columns={
                time_col: "timestamp",
                "Open": "open",
                "High": "high",
                "Low": "low",
                "Close": "close"
            }
        )


        needed = [
            "timestamp",
            "open",
            "high",
            "low",
            "close"
        ]


        df = df[
            needed
        ].copy()


        chunks.append(
            df
        )


        print(
            f"  {len(df):,} bars"
        )


        cursor = chunk_end


    if len(chunks) == 0:

        return pd.DataFrame()


    return pd.concat(
        chunks,
        ignore_index=True
    )


# ================================================================================================
# NORMALIZE TIMESTAMPS
# ================================================================================================

def normalize_m1(df):

    x = df.copy()


    # DST-safe parsing.
    #
    # Cache files may eventually contain timestamps from both EDT and EST,
    # so normalize through UTC before converting back to New York.

    x["timestamp"] = pd.to_datetime(
        x["timestamp"],
        utc=True,
        errors="coerce"
    )


    x = x.dropna(
        subset=["timestamp"]
    )


    x["timestamp"] = (
        x["timestamp"]
        .dt.tz_convert(
            NY_TZ
        )
    )


    for c in [
        "open",
        "high",
        "low",
        "close"
    ]:

        x[c] = pd.to_numeric(
            x[c],
            errors="coerce"
        )


    x = x.dropna(
        subset=[
            "open",
            "high",
            "low",
            "close"
        ]
    )


    x = (
        x.sort_values(
            "timestamp"
        )
        .drop_duplicates(
            subset=["timestamp"],
            keep="last"
        )
        .reset_index(
            drop=True
        )
    )


    x["date"] = (
        x["timestamp"].dt.date
    )


    x["minute_of_day"] = (

        x["timestamp"].dt.hour * 60

        +

        x["timestamp"].dt.minute

    )


    x = x[
        (x["minute_of_day"] >= SESSION_START)
        &
        (x["minute_of_day"] <= SESSION_END)
    ].copy()


    return x


# ================================================================================================
# LOAD EXISTING CACHE
# ================================================================================================

def load_cache():

    if not CACHE_FILE.exists():

        return pd.DataFrame()


    try:

        old = pd.read_csv(
            CACHE_FILE
        )


        if len(old) == 0:

            return pd.DataFrame()


        # Leave timestamp parsing to normalize_m1().
        # This is safer when the cache spans DST transitions.

        return old


    except Exception as e:

        print(
            f"Cache read warning: {e}"
        )

        return pd.DataFrame()


# ================================================================================================
# MERGE CACHE + NEW DOWNLOAD
# ================================================================================================

def update_cache():

    old = load_cache()

    new = download_yahoo_m1()


    pieces = []


    if len(old):

        pieces.append(
            old
        )


    if len(new):

        pieces.append(
            new
        )


    if len(pieces) == 0:

        raise RuntimeError(
            "No SPX M1 data available."
        )


    combined = pd.concat(
        pieces,
        ignore_index=True
    )


    combined = normalize_m1(
        combined
    )


    save = combined[
        [
            "timestamp",
            "open",
            "high",
            "low",
            "close"
        ]
    ].copy()


    save["timestamp"] = (
        save["timestamp"]
        .astype(str)
    )


    save.to_csv(
        CACHE_FILE,
        index=False
    )


    return combined


# ================================================================================================
# AGGREGATE BARS
# ================================================================================================

def aggregate_bars(
    source,
    minutes,
    anchor_minute
):

    x = source.copy()


    x["bucket"] = np.floor_divide(
        x["minute_of_day"]
        -
        anchor_minute,
        minutes
    )


    bars = (

        x.groupby(
            [
                "date",
                "bucket"
            ],
            as_index=False
        )

        .agg(
            open=("open", "first"),
            high=("high", "max"),
            low=("low", "min"),
            close=("close", "last"),
            count=("close", "size")
        )

        .sort_values(
            [
                "date",
                "bucket"
            ]
        )

        .reset_index(
            drop=True
        )

    )


    return bars


# ================================================================================================
# M1 EMA20
# ================================================================================================

def add_m1_ema20(m1):

    x = m1.copy()


    x["EMA20_M1"] = (

        x["close"]

        .ewm(
            span=20,
            adjust=False
        )

        .mean()

    )


    return x


# ================================================================================================
# ATR14
# ================================================================================================

def build_atr14_map(m1):

    source = m1[
        (m1["minute_of_day"] >= ATR_SESSION_START)
        &
        (m1["minute_of_day"] <= ATR_SESSION_END)
    ].copy()


    daily = (

        source.groupby(
            "date",
            as_index=False
        )

        .agg(
            open=("open", "first"),
            high=("high", "max"),
            low=("low", "min"),
            close=("close", "last"),
            bars=("close", "size")
        )

        .sort_values(
            "date"
        )

        .reset_index(
            drop=True
        )

    )


    daily["prev_close"] = (
        daily["close"]
        .shift(1)
    )


    daily["tr"] = pd.concat(
        [

            daily["high"]
            -
            daily["low"],

            (
                daily["high"]
                -
                daily["prev_close"]
            ).abs(),

            (
                daily["low"]
                -
                daily["prev_close"]
            ).abs()

        ],
        axis=1
    ).max(
        axis=1
    )


    daily["atr14"] = (

        daily["tr"]

        .rolling(
            14,
            min_periods=14
        )

        .mean()

        .shift(1)

    )


    atr_map = {

        row["date"]:
            float(
                row["atr14"]
            )

        for _, row in daily.iterrows()

        if pd.notna(
            row["atr14"]
        )

    }


    return atr_map, daily


# ================================================================================================
# HISTORY AVAILABLE AT SIGNAL TIME
# ================================================================================================

def history_at_snapshot(
    m1,
    target_date
):

    old = m1[
        m1["date"]
        <
        target_date
    ].copy()


    today = m1[
        (m1["date"] == target_date)
        &
        (m1["minute_of_day"] <= SIGNAL_LAST_MINUTE)
    ].copy()


    return pd.concat(
        [
            old,
            today
        ],
        ignore_index=True
    )


# ================================================================================================
# FEATURES
# ================================================================================================

def compute_features(
    m1,
    atr_map,
    target_date
):

    morning = m1[
        (m1["date"] == target_date)
        &
        (m1["minute_of_day"] >= SESSION_START)
        &
        (m1["minute_of_day"] <= SIGNAL_LAST_MINUTE)
    ].copy()


    if len(morning) == 0:

        raise RuntimeError(
            f"No morning data for {target_date}"
        )


    expected = 149


    if len(morning) != expected:

        print()

        print(
            f"WARNING: Expected {expected} bars "
            f"09:30-11:58; received {len(morning)}."
        )


    if (
        int(
            morning.iloc[-1]["minute_of_day"]
        )
        !=
        SIGNAL_LAST_MINUTE
    ):

        raise RuntimeError(
            "11:58 ET bar is unavailable. "
            "Cannot reproduce frozen signal."
        )


    current = morning.iloc[-1]


    close = float(
        current["close"]
    )


    morning_open = float(
        morning.iloc[0]["open"]
    )


    morning_high = float(
        morning["high"].max()
    )


    morning_low = float(
        morning["low"].min()
    )


    morning_return = (
        close
        -
        morning_open
    )


    morning_range = (
        morning_high
        -
        morning_low
    )


    if morning_range <= 0:

        raise RuntimeError(
            "Morning range is zero or invalid."
        )


    range_position = (

        (
            close
            -
            morning_low
        )

        /

        morning_range

    )


    atr14 = atr_map.get(
        target_date
    )


    if atr14 is None:

        raise RuntimeError(
            "ATR14 unavailable. Need at least "
            "15 completed cached trading sessions."
        )


    ema20_m1 = float(
        current["EMA20_M1"]
    )


    dist_high_atr = (

        (
            morning_high
            -
            close
        )

        /

        atr14

    )


    ema20_dist_atr = (

        (
            close
            -
            ema20_m1
        )

        /

        atr14

    )


    history = history_at_snapshot(
        m1,
        target_date
    )


    # ============================================================================================
    # 10 MINUTE
    # ============================================================================================

    bars10 = aggregate_bars(
        history,
        10,
        570
    )


    c10 = (

        bars10["close"]

        .astype(float)

        .reset_index(
            drop=True
        )

    )


    if len(c10) < 50:

        raise RuntimeError(
            "Insufficient 10-minute history for EMA50."
        )


    ema20 = (

        c10

        .ewm(
            span=20,
            adjust=False
        )

        .mean()

        .iloc[-1]

    )


    ema50 = (

        c10

        .ewm(
            span=50,
            adjust=False
        )

        .mean()

        .iloc[-1]

    )


    ema20_50_10m = float(
        ema20
        -
        ema50
    )


    if len(c10) < 20:

        raise RuntimeError(
            "Insufficient 10-minute history for REG20."
        )


    reg20_10m = regression_slope(
        c10.iloc[-20:]
    )


    # ============================================================================================
    # 15 MINUTE ROC
    # ============================================================================================

    bars15 = aggregate_bars(
        history,
        15,
        569
    )


    c15 = (

        bars15["close"]

        .astype(float)

        .reset_index(
            drop=True
        )

    )


    if len(c15) < 6:

        raise RuntimeError(
            "Insufficient 15-minute history for ROC5."
        )


    roc5_15m = float(
        c15.iloc[-1]
        -
        c15.iloc[-6]
    )


    # ============================================================================================
    # 30 MINUTE ROC
    # ============================================================================================

    bars30 = aggregate_bars(
        history,
        30,
        569
    )


    c30 = (

        bars30["close"]

        .astype(float)

        .reset_index(
            drop=True
        )

    )


    if len(c30) < 4:

        raise RuntimeError(
            "Insufficient 30-minute history for ROC3."
        )


    roc3_30m = float(
        c30.iloc[-1]
        -
        c30.iloc[-4]
    )


    return {

        "CLOSE":
            close,

        "MORNING_OPEN":
            morning_open,

        "MORNING_HIGH":
            morning_high,

        "MORNING_LOW":
            morning_low,

        "MORNING_RETURN":
            morning_return,

        "RANGE_POSITION":
            range_position,

        "EMA20_50_10M":
            ema20_50_10m,

        "REG20_10M":
            reg20_10m,

        "ROC5_15M":
            roc5_15m,

        "ROC3_30M":
            roc3_30m,

        "ATR14":
            atr14,

        "EMA20_M1":
            ema20_m1,

        "DIST_HIGH_ATR":
            dist_high_atr,

        "EMA20_DIST_ATR":
            ema20_dist_atr

    }


# ================================================================================================
# DECISION
# ================================================================================================

def make_decision(f):

    vote_details = {

        "EMA10":
            sign_value(
                f["EMA20_50_10M"]
            ),

        "REG10":
            sign_value(
                f["REG20_10M"]
            ),

        "ROC15":
            sign_value(
                f["ROC5_15M"]
            ),

        "ROC30":
            sign_value(
                f["ROC3_30M"]
            ),

        "MORNING":
            sign_value(
                f["MORNING_RETURN"]
            )

    }


    votes = sum(
        vote_details.values()
    )


    if votes > 0:

        base_dir = 1

    elif votes < 0:

        base_dir = -1

    else:

        base_dir = 0


    pos_n = (

        (
            f["RANGE_POSITION"]
            -
            0.5
        )

        *
        2

    )


    trade_allowed = (

        base_dir != 0

        and

        abs(pos_n)
        <=
        POSITION_LIMIT

    )


    correction = (

        base_dir == -1

        and

        trade_allowed

        and

        f["DIST_HIGH_ATR"]
        >=
        DIST_HIGH_THRESHOLD

        and

        f["EMA20_DIST_ATR"]
        <=
        EMA20_DIST_THRESHOLD

    )


    if not trade_allowed:

        action = "NO_TRADE"

    elif correction:

        action = "BULLISH"

    elif base_dir > 0:

        action = "BULLISH"

    else:

        action = "BEARISH"


    return {

        "VOTE_DETAILS":
            vote_details,

        "VOTES":
            votes,

        "BASE_DIR":
            base_dir,

        "POSITION_NORMALIZED":
            pos_n,

        "TRADE_ALLOWED":
            trade_allowed,

        "CORRECTION":
            correction,

        "ACTION":
            action

    }


# ================================================================================================
# DOWNLOAD + CACHE
# ================================================================================================

m1 = update_cache()


print()
print("=" * 110)
print("CACHE SUMMARY")
print("=" * 110)


print(
    f"Bars:          {len(m1):,}"
)

print(
    f"Trading days:  {m1['date'].nunique()}"
)

print(
    f"First:         {m1['timestamp'].min()}"
)

print(
    f"Last:          {m1['timestamp'].max()}"
)


# ================================================================================================
# ADD EMA
# ================================================================================================

m1 = add_m1_ema20(
    m1
)


# ================================================================================================
# ATR
# ================================================================================================

atr_map, daily = build_atr14_map(
    m1
)


# ================================================================================================
# DETERMINE TARGET DATE
# ================================================================================================

now_ny = pd.Timestamp.now(
    tz=NY_TZ
)


today_ny = now_ny.date()


today_has_signal = (

    (
        (m1["date"] == today_ny)
        &
        (
            m1["minute_of_day"]
            ==
            SIGNAL_LAST_MINUTE
        )
    ).any()

)


if today_has_signal:

    target_date = today_ny

else:

    eligible = m1[
        m1["minute_of_day"]
        ==
        SIGNAL_LAST_MINUTE
    ]["date"]


    if len(eligible) == 0:

        raise RuntimeError(
            "No 11:58 ET SPX bar found."
        )


    target_date = max(
        eligible
    )


    print()

    print(
        "WARNING: Today's 11:58 ET bar was not found."
    )

    print(
        f"Using most recent available trading date: "
        f"{target_date}"
    )


# ================================================================================================
# DATA QUALITY
# ================================================================================================

target_session = m1[
    m1["date"]
    ==
    target_date
].copy()


target_morning = target_session[
    (target_session["minute_of_day"] >= 570)
    &
    (target_session["minute_of_day"] <= 718)
]


print()
print("=" * 110)
print("TARGET DATA QUALITY")
print("=" * 110)


print(
    f"Target date:            {target_date}"
)

print(
    f"Full session bars:      {len(target_session)}"
)

print(
    f"09:30-11:58 bars:       {len(target_morning)} / 149"
)

print(
    f"First target bar:       "
    f"{target_session['timestamp'].min()}"
)

print(
    f"Last target bar:        "
    f"{target_session['timestamp'].max()}"
)


# ================================================================================================
# SIGNAL
# ================================================================================================

features = compute_features(
    m1,
    atr_map,
    target_date
)


decision = make_decision(
    features
)


action = decision[
    "ACTION"
]


if action == "BULLISH":

    options_action = (
        "SELL PUT CREDIT SPREAD"
    )

    decision_color = (
        "#00A651"
    )

    decision_background = (
        "#EAF8EF"
    )


elif action == "BEARISH":

    options_action = (
        "SELL CALL CREDIT SPREAD"
    )

    decision_color = (
        "#D32F2F"
    )

    decision_background = (
        "#FDECEC"
    )


else:

    options_action = (
        "NO TRADE"
    )

    decision_color = (
        "#777777"
    )

    decision_background = (
        "#F2F2F2"
    )


# ================================================================================================
# LARGE VISUAL DECISION
# ================================================================================================

decision_html = f"""
<div style="
    font-family: Arial, Helvetica, sans-serif;
    max-width: 950px;
    margin-top: 20px;
    margin-bottom: 20px;
">

    <div style="
        border: 3px solid {decision_color};
        background: {decision_background};
        border-radius: 14px;
        padding: 24px;
        text-align: center;
    ">

        <div style="
            font-size: 16px;
            font-weight: 700;
            color: #444;
            margin-bottom: 8px;
        ">
            SPX TRADE DECISION — {target_date}
        </div>

        <div style="
            font-size: 42px;
            line-height: 1.1;
            font-weight: 900;
            color: {decision_color};
        ">
            {action.replace("_", " ")}
        </div>

        <div style="
            font-size: 23px;
            font-weight: 900;
            color: {decision_color};
            margin-top: 10px;
        ">
            {options_action}
        </div>

        <div style="
            font-size: 13px;
            color: #666;
            margin-top: 12px;
        ">
            Frozen snapshot: 11:58 ET
        </div>

    </div>

</div>
"""


display(
    HTML(
        decision_html
    )
)


# ================================================================================================
# SIGNAL DETAILS
# ================================================================================================

print()
print("=" * 110)
print("SPX NOON DIRECTION SIGNAL")
print("=" * 110)


print(
    f"Date:                   {target_date}"
)

print(
    "Frozen snapshot:        11:58 ET"
)

print(
    "Signal usable:          after 11:58 ET bar closes"
)


print()
print(
    f"DIRECTION:              {action}"
)

print(
    f"OPTIONS ACTION:         {options_action}"
)


print()
print("-" * 110)


print(
    f"Votes:                  {decision['VOTES']:+d}"
)

print(
    f"EMA 10m vote:           {decision['VOTE_DETAILS']['EMA10']:+d}"
)

print(
    f"REG 10m vote:           {decision['VOTE_DETAILS']['REG10']:+d}"
)

print(
    f"ROC 15m vote:           {decision['VOTE_DETAILS']['ROC15']:+d}"
)

print(
    f"ROC 30m vote:           {decision['VOTE_DETAILS']['ROC30']:+d}"
)

print(
    f"Morning-return vote:    {decision['VOTE_DETAILS']['MORNING']:+d}"
)


print()
print("-" * 110)


print(
    f"11:58 SPX close:        {features['CLOSE']:.2f}"
)

print(
    f"Morning return:         {features['MORNING_RETURN']:+.2f}"
)

print(
    f"Range position:         {features['RANGE_POSITION']:.6f}"
)

print(
    f"ATR14:                  {features['ATR14']:.6f}"
)

print(
    f"DIST_HIGH_ATR:          {features['DIST_HIGH_ATR']:.8f}"
)

print(
    f"EMA20_DIST_ATR:         {features['EMA20_DIST_ATR']:.8f}"
)


print()
print("-" * 110)


print(
    f"Trade allowed:          {decision['TRADE_ALLOWED']}"
)

print(
    f"Bearish correction:     {decision['CORRECTION']}"
)


print()
print("=" * 110)


# ================================================================================================
# FROZEN VALIDATION DASHBOARD
# ================================================================================================

validation_html = f"""
<div style="
    font-family: Arial, Helvetica, sans-serif;
    max-width: 950px;
    border: 1px solid #cccccc;
    border-radius: 12px;
    padding: 20px;
    margin-top: 20px;
">

    <div style="
        font-size: 21px;
        font-weight: 900;
        margin-bottom: 5px;
    ">
        FROZEN HISTORICAL VALIDATION
    </div>

    <div style="
        font-size: 13px;
        color: #666;
        margin-bottom: 17px;
    ">
        Reminder of the historical evidence behind the production engine.
        These numbers are fixed and are not recalculated from today's signal.
    </div>

    <table style="
        width:100%;
        border-collapse:collapse;
        font-size:14px;
    ">

        <tr>
            <td style="padding:8px;"><b>Validation Period</b></td>
            <td style="padding:8px;">{VALIDATION["period"]}</td>
            <td style="padding:8px;"><b>Dates Tested</b></td>
            <td style="padding:8px;"><b>{VALIDATION["dates"]}</b></td>
        </tr>

        <tr style="background:#f5f5f5;">
            <td style="padding:8px;"><b>Actual Trades</b></td>
            <td style="padding:8px;"><b>{VALIDATION["trades"]}</b></td>
            <td style="padding:8px;"><b>Wins / Losses</b></td>
            <td style="padding:8px;"><b>{VALIDATION["wins"]} / {VALIDATION["losses"]}</b></td>
        </tr>

        <tr>
            <td style="padding:8px;"><b>Historical Win Rate</b></td>
            <td style="padding:8px;"><b>{VALIDATION["win_rate"]:.2f}%</b></td>
            <td style="padding:8px;"><b>Profit Factor</b></td>
            <td style="padding:8px;"><b>{VALIDATION["profit_factor"]:.2f}</b></td>
        </tr>

        <tr style="background:#f5f5f5;">
            <td style="padding:8px;"><b>Historical Net</b></td>
            <td style="padding:8px;"><b>${VALIDATION["net_profit"]:,.2f}</b></td>
            <td style="padding:8px;"><b>Max Drawdown</b></td>
            <td style="padding:8px;"><b>${VALIDATION["max_drawdown"]:,.2f}</b></td>
        </tr>

        <tr>
            <td style="padding:8px;"><b>Bullish Signals</b></td>
            <td style="padding:8px;">{VALIDATION["bullish"]}</td>
            <td style="padding:8px;"><b>Bearish Signals</b></td>
            <td style="padding:8px;">{VALIDATION["bearish"]}</td>
        </tr>

        <tr style="background:#f5f5f5;">
            <td style="padding:8px;"><b>No-Trade Signals</b></td>
            <td style="padding:8px;">{VALIDATION["no_trade"]}</td>
            <td style="padding:8px;"><b>Production Reconstruction</b></td>
            <td style="padding:8px;"><b>{VALIDATION["reconstruction"]}</b></td>
        </tr>

    </table>


    <div style="
        font-size:17px;
        font-weight:900;
        margin-top:23px;
        margin-bottom:8px;
    ">
        SHORT-HISTORY / YAHOO WARM-UP ROBUSTNESS
    </div>


    <table style="
        width:100%;
        border-collapse:collapse;
        font-size:14px;
    ">

        <tr style="background:#eeeeee;">
            <th style="padding:8px;text-align:left;">
                Previous Trading Sessions
            </th>

            <th style="padding:8px;text-align:left;">
                Final Decision Agreement
            </th>
        </tr>

        <tr>
            <td style="padding:8px;">21</td>
            <td style="padding:8px;"><b>{VALIDATION["warmup_21"]}</b></td>
        </tr>

        <tr style="background:#f5f5f5;">
            <td style="padding:8px;">18</td>
            <td style="padding:8px;"><b>{VALIDATION["warmup_18"]}</b></td>
        </tr>

        <tr>
            <td style="padding:8px;">16</td>
            <td style="padding:8px;"><b>{VALIDATION["warmup_16"]}</b></td>
        </tr>

        <tr style="background:#f5f5f5;">
            <td style="padding:8px;">15</td>
            <td style="padding:8px;"><b>{VALIDATION["warmup_15"]}</b></td>
        </tr>

    </table>


    <div style="
        font-size:12px;
        color:#777777;
        margin-top:16px;
    ">
        Historical results are descriptive and do not guarantee future performance.
        Frozen strategy thresholds are not modified by this production script.
    </div>

</div>
"""


display(
    HTML(
        validation_html
    )
)


# ================================================================================================
# SAVE TODAY'S SIGNAL
# ================================================================================================

signal_file = (
    CACHE_DIR /
    "SPX_NOON_SIGNALS.csv"
)


signal_row = pd.DataFrame(
    [
        {

            "date":
                target_date,

            "generated_at_et":
                str(now_ny),

            "direction":
                action,

            "options_action":
                options_action,

            "votes":
                decision["VOTES"],

            "trade_allowed":
                decision["TRADE_ALLOWED"],

            "correction":
                decision["CORRECTION"],

            **{
                k: v
                for k, v in features.items()
            }

        }
    ]
)


if signal_file.exists():

    old_signals = pd.read_csv(
        signal_file
    )


    old_signals = old_signals[
        old_signals["date"].astype(str)
        !=
        str(target_date)
    ]


    signal_row = pd.concat(
        [
            old_signals,
            signal_row
        ],
        ignore_index=True
    )


signal_row.to_csv(
    signal_file,
    index=False
)


print(
    f"\nSignal history saved to:\n{signal_file}"
)


print()

print(
    "IMPORTANT: This signal uses Yahoo ^GSPC M1 data. "
    "The frozen engine itself is unchanged."
)